In [1]:
%pip install -r "../requirements.txt"

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer
from datasets import load_dataset

/home/azureuser/repositories/proai/course9-genai/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
DATASET_NAME = "SetFit/tweet_sentiment_extraction"
DEVICE = torch.device("cuda")

In [4]:
dataset = load_dataset(DATASET_NAME)

Repo card metadata block was not found. Setting CardData to empty.


In [5]:
sentiments = set(dataset['train']['label_text'])
sentiments

{'negative', 'neutral', 'positive'}

In [6]:
dataset['train'][:5]

{'textID': ['cb774db0d1',
  '549e992a42',
  '088c60f138',
  '9642c003ef',
  '358bd9e861'],
 'text': [' I`d have responded, if I were going',
  ' Sooo SAD I will miss you here in San Diego!!!',
  'my boss is bullying me...',
  ' what interview! leave me alone',
  ' Sons of ****, why couldn`t they put them on the releases we already bought'],
 'label': [1, 0, 0, 0, 0],
 'label_text': ['neutral', 'negative', 'negative', 'negative', 'negative']}

In [7]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME,
                                             torch_dtype="auto").to(DEVICE)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME,
                                          torch_dtype="auto")
streamer = TextStreamer(tokenizer)

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 103.15it/s]



In [8]:
model

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLU()
        )
        (input_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (norm): Phi3RMSNorm((3072,), eps=1e-05)
    (rotary_emb): Phi3RotaryEmbedding()
  )
  (lm_head): Linear(in_features=3072, out_features=32064, 

In [ ]:
input_text = "Test sentiment normal"

prompt = f'''
Classify the following sentence in {sentiments} \
Return only the classification of the sentence provided below.

Text: {input_text}
Answer:
'''

In [37]:
tokenized_text = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(DEVICE)
tokenized_output = model.generate(**tokenized_text, max_new_tokens=2, do_sample=False)
output = tokenizer.batch_decode(tokenized_output, skip_special_tokens=True)[0].replace(prompt, "")

In [38]:
output

'\n\n'